[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mnrozhkov/santa-scale-challenge/blob/serverless/notebooks/03_text_to_music.ipynb)

# Text → music (ACE-Step)

Playground for the **audio** role: ACE-Step via `adapter_for("audio", settings)`.
`generate(prompt, mood=...)` returns MP3 bytes. The mood tag is how Santa picks a
track; the prompt itself comes from `config/prompts.yaml` (`santa.animate.music_prompt`).

**Fallback** (`SANTA_FALLBACK`, default `auto`): try ACE-Step first; if the endpoint
is missing or fails, play a bundled `local_tracks` mp3 from `data/fallback/moods/`.
`off` / `only` as usual. Audio fallback is local files, not OpenAI.

**Secrets.** Locally `.env`; on Colab userdata named like `.env.example`:
`AUDIO_ENDPOINT_URL`, `AUDIO_ENDPOINT_TOKEN`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys


def apply_colab_secrets() -> None:
    """Copy Colab userdata into os.environ. Secret names match .env.example."""
    try:
        from google.colab import userdata
    except ImportError:
        return
    for key in (
        "IMAGE_ENDPOINT_URL",
        "IMAGE_ENDPOINT_TOKEN",
        "VIDEO_ENDPOINT_URL",
        "VIDEO_ENDPOINT_TOKEN",
        "AUDIO_ENDPOINT_URL",
        "AUDIO_ENDPOINT_TOKEN",
        "TOKEN_FACTORY_API_KEY",
        "OPENAI_API_KEY",
        "SANTA_FALLBACK",
    ):
        try:
            value = userdata.get(key)
        except Exception:
            continue
        if value:
            os.environ[key] = str(value)


try:
    import google.colab  # noqa: F401
except ImportError:
    pass
else:
    apply_colab_secrets()
    repo = Path("/content/santa-scale-challenge")
    if not (repo / "pyproject.toml").is_file():
        subprocess.check_call(
            [
                "git",
                "clone",
                "--branch",
                "serverless",
                "--depth",
                "1",
                "https://github.com/mnrozhkov/santa-scale-challenge.git",
                str(repo),
            ]
        )
    os.chdir(repo)
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo)])

from santa.config import Settings

settings = Settings.load()
print("SANTA_FALLBACK =", os.environ.get("SANTA_FALLBACK", "auto"))
print(
    "env present:",
    [
        k
        for k in (
            "IMAGE_ENDPOINT_URL",
            "IMAGE_ENDPOINT_TOKEN",
            "VIDEO_ENDPOINT_URL",
            "VIDEO_ENDPOINT_TOKEN",
            "AUDIO_ENDPOINT_URL",
            "AUDIO_ENDPOINT_TOKEN",
            "TOKEN_FACTORY_API_KEY",
            "OPENAI_API_KEY",
        )
        if os.environ.get(k)
    ],
)


## Parameters

- **`mood`** — one of `santa.prompts.MOODS` (dropdown in the next cell, or set the variable).
- **music prompt** — `config/prompts.yaml` `music.<mood>`; override with `PROMPT`.
- **`audio_duration`**, **`inference_steps`**, **`audio_format`** — `roles.audio.options`
  in yaml, on `Settings`. Example: `audio.cfg.options["audio_duration"] = 8`.


In [ ]:
from santa.prompts import MOODS

print("MOODS:", ", ".join(MOODS))
mood = "warm"

try:
    import ipywidgets as widgets
    from IPython.display import display

    mood_picker = widgets.Dropdown(options=list(MOODS), value=mood, description="mood")
    display(mood_picker)
except ImportError:
    mood_picker = None
    print("ipywidgets not installed — edit mood =", mood)


In [ ]:
from IPython.display import Audio, display
from santa.animate import load_prompts, music_prompt
from santa.models import adapter_for

if mood_picker is not None:
    mood = mood_picker.value

PROMPT = None  # or a custom string; None → prompts.yaml for this mood
# audio_duration lives in Settings / models.yaml options (default 8)

prompts = load_prompts()
prompt = PROMPT or music_prompt(prompts, mood)
audio = adapter_for("audio", settings)
# audio.cfg.options["audio_duration"] = 8

mp3 = audio.generate(prompt, mood=mood)
track = Path("out/notebook_t2m.mp3")
track.parent.mkdir(parents=True, exist_ok=True)
track.write_bytes(mp3)

print(audio.cfg.label, "mood=", mood, "fallback=", audio.used_fallback)
display(Audio(data=mp3, autoplay=False))
